In [1]:
!pip install 'dask[complete]' sqlalchemy psycopg2-binary pandas

In [2]:
import dask.dataframe as dd
import pandas as pd
from sqlalchemy import create_engine  # Cet import manquait
from datetime import datetime

In [3]:
# Paramètres de connexion PostgreSQL
db_config = {
    'user': 'tchap_stats_2026',        # Remplace par ton nom d'utilisateur PostgreSQL
    'password': 'uTs-Lzz9i7u6YyMlq3QH',    # Remplace par ton mot de passe PostgreSQL
    'host': '127.0.0.1',            # Remplace par ton hôte PostgreSQL
    'port': '10000',            # Remplace par le port de ton serveur PostgreSQL
    'dbname': 'tchap_stats_2026'       # Remplace par le nom de ta base de données
}

# Construire l'URL de connexion PostgreSQL
db_url = f"postgresql://{db_config['user']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['dbname']}"

# Créer une connexion à PostgreSQL via SQLAlchemy
engine = create_engine(db_url)

In [4]:
# Lire les données par morceaux avec Pandas (en utilisant `chunksize` pour gérer les grands volumes de données)
table_name = 'user_daily_visits'  # Remplace par le nom de ta table
query = f'SELECT * FROM {table_name}'

# Lire les données par morceaux (chunksize = nombre de lignes par morceau)
chunksize = 250000  # Ajuster selon la mémoire disponible
chunks = pd.read_sql(query, con=engine, chunksize=chunksize)

# Convertir chaque chunk en DataFrame Dask et concaténer
ddf = dd.concat([dd.from_pandas(chunk, npartitions=10) for chunk in chunks])


In [5]:
ddf.head()

,user_id,device_id,visit_ts,user_agent,instance,domain
0,\x979530afff7a11e9628134a4f4f034f3cbb678326017...,\x101e4cca91f20ae22dcbb8fd5fe5437bdeb5e4998198...,2022-04-11 22:00:00+00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,agent_e,protonmail.com
1,\x979530afff7a11e9628134a4f4f034f3cbb678326017...,\xa4100af421a0b69f382599824cc918e270c570cc7d8a...,2022-04-11 22:00:00+00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,agent_e,protonmail.com
2,\x979530afff7a11e9628134a4f4f034f3cbb678326017...,\x9b5870b30a0c8d8de3f5adaa802459e6db0e0a9309e1...,2022-04-11 22:00:00+00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,agent_e,protonmail.com
3,\x979530afff7a11e9628134a4f4f034f3cbb678326017...,\x0daa39c526de06f9909eafab96386d222316cfb4f53e...,2022-04-11 22:00:00+00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,agent_e,protonmail.com
4,\x979530afff7a11e9628134a4f4f034f3cbb678326017...,\x56e562c80bfdc6fb0067b1847e144566680dc565cab7...,2022-04-11 22:00:00+00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,agent_e,protonmail.com


In [6]:
# Compter le nombre total de lignes (Dask gère ça de manière paresseuse)
count = ddf.shape[0].compute()
print(f"Nombre total de lignes : {count}")

Nombre total de lignes : 2496


In [7]:

import re

def map_device_type(user_agent):
    if not isinstance(user_agent, str):
        return 'Autre'  # Ou une autre valeur par défaut pour les valeurs non valides
    if re.search(r'Mozilla.*Windows.*Firefox', user_agent):
        return 'Firefox Windows'
    elif re.search(r'Mozilla.*Windows.*Chrome', user_agent):
        return 'Chrome Windows'
    elif re.search(r'Mozilla.*Windows.*Trident', user_agent):
        return 'Internet Explorer Windows'
    elif re.search(r'Mozilla.*Mac OS.*Firefox', user_agent):
        return 'Firefox Mac OS'
    elif re.search(r'Mozilla.*Mac OS.*Chrome', user_agent):
        return 'Chrome Mac OS'
    elif re.search(r'Mozilla.*Mac OS.*Safari', user_agent):
        return 'Safari Mac OS'
    elif re.search(r'Mozilla.*Android.*Firefox', user_agent):
        return 'Firefox Android'
    elif re.search(r'Mozilla.*Android.*Chrome', user_agent):
        return 'Chrome Android'
    elif re.search(r'Mozilla.*Linux.*Firefox', user_agent):
        return 'Firefox Linux'
    elif re.search(r'Mozilla.*Linux.*Chrome', user_agent):
        return 'Chrome Linux'
    elif re.search(r'Mozilla.*CrOS.*Chrome', user_agent):
        return 'Chrome OS'
    elif re.search(r'Mozilla.*Mobile', user_agent):
        return 'Navigateur Mobile'
    elif re.search(r'Mozilla', user_agent):
        return 'Autre Navigateur'
    elif re.search(r'Tchap.*NEO', user_agent):
        return 'Tchap Android NEO'
    elif re.search(r'Tchap.*Android', user_agent):
        return 'Tchap Android'
    elif re.search(r'RiotNSE/2.*iOS', user_agent):
        return 'Tchap iOS'
    elif re.search(r'RiotSharedExtension/2.*iOS', user_agent):
        return 'Tchap iOS'
    elif re.search(r'Tchap.*iOS', user_agent):
        return 'Tchap iOS'
    elif re.search(r'Riot', user_agent):
        return 'Element'
    elif re.search(r'Element', user_agent):
        return 'Element'
    else:
        return 'Autre'
        return 'Autre'

def map_platform(user_agent):
    if not isinstance(user_agent, str):
        return 'Autre'  # Ou une autre valeur par défaut

    if re.search(r'Mozilla', user_agent):
        return 'Web'
    elif re.search(r'Tchap.*Android', user_agent) or re.search(r'Riot.*iOS', user_agent):
        return 'Mobile'
    elif re.search(r'Element.*iOS', user_agent) or re.search(r'Element.*Android', user_agent):
        return 'Mobile'
    else:
        return 'Autre'

In [8]:
ddf.head()

,user_id,device_id,visit_ts,user_agent,instance,domain
0,\x979530afff7a11e9628134a4f4f034f3cbb678326017...,\x101e4cca91f20ae22dcbb8fd5fe5437bdeb5e4998198...,2022-04-11 22:00:00+00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,agent_e,protonmail.com
1,\x979530afff7a11e9628134a4f4f034f3cbb678326017...,\xa4100af421a0b69f382599824cc918e270c570cc7d8a...,2022-04-11 22:00:00+00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,agent_e,protonmail.com
2,\x979530afff7a11e9628134a4f4f034f3cbb678326017...,\x9b5870b30a0c8d8de3f5adaa802459e6db0e0a9309e1...,2022-04-11 22:00:00+00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,agent_e,protonmail.com
3,\x979530afff7a11e9628134a4f4f034f3cbb678326017...,\x0daa39c526de06f9909eafab96386d222316cfb4f53e...,2022-04-11 22:00:00+00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,agent_e,protonmail.com
4,\x979530afff7a11e9628134a4f4f034f3cbb678326017...,\x56e562c80bfdc6fb0067b1847e144566680dc565cab7...,2022-04-11 22:00:00+00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,agent_e,protonmail.com


In [9]:
# 1. Appliquer les fonctions de mappage pour créer les colonnes 'device_type' et 'platform'
ddf['device_type'] = ddf['user_agent'].map(map_device_type, meta=('device_type', 'str'))
ddf['platform'] = ddf['user_agent'].map(map_platform, meta=('platform', 'str'))

# 2. Supprimer la timezone de 'visit_ts' (rendre tz-naive)
ddf['visit_ts'] = ddf['visit_ts'].dt.tz_convert(None)

# 3. Créer la colonne 'month' en tronquant 'visit_ts' à la période mensuelle (sans timezone)
ddf['month'] = ddf['visit_ts'].dt.to_period('M')

# 4. Créer un Timestamp pour 18 mois dans le passé
from pandas.tseries.offsets import DateOffset
eighteen_months_ago = pd.Timestamp.now() - DateOffset(months=18)

# 5. Filtrer les données pour les 18 derniers mois
ddf_filtered = ddf[ddf['visit_ts'] >= eighteen_months_ago]

# 6. Afficher un aperçu des données filtrées
ddf_filtered.compute()

# 7. Grouper par les colonnes 'device_id', 'month', 'user_id', 'instance', 'domain', 'device_type', 'platform'
ddf_grouped = ddf_filtered.groupby(
    ['device_id', 'month', 'user_id', 'instance', 'domain', 'device_type', 'platform']
).size().reset_index()

# Renommer la colonne résultante de 'size' à 'visit_count'
ddf_grouped = ddf_grouped.rename(columns={0: 'visit_count'})

# 8. Afficher un aperçu des données groupées
ddf_grouped.compute()
ddf_grouped.head()


,device_id,month,user_id,instance,domain,device_type,platform,visit_count
0,\x0752229a6a4cbef08033a28fc9e3d7b44e3dbcbed3ac...,2023-10,\x54b836e71c9e694aecd15d64cb39df4e03cd312ac1ba...,preprod_ext01,net.com,Autre,Autre,8
1,\x1d63e53ad94d39da1aad65d8f9324eac881e53c808f8...,2023-10,\x5e53ee508b59830af259cf7546af4d7aae8d274736b2...,preprod_ext01,preprod.tchap.beta.gouv.fr,Autre,Autre,1
2,\xd7a439487be65f9b08b8193b16c36450f636296fcaeb...,2023-10,\x54b836e71c9e694aecd15d64cb39df4e03cd312ac1ba...,preprod_ext01,net.com,Firefox Mac OS,Web,1
3,\x0079856f94bfb9300a595b3fc11cc0eb6180a5b79428...,2023-09,\xfe71bedcea760ae910adde760e946d2ecd9d89708899...,preprod_int01,beta.gouv.fr,Autre,Autre,1
4,\x07bfa1a5b56285fb3130c9fbaec6d20ccf8d2983783d...,2023-10,\x4a2ac614087b08a55db318f8b00216313c6ed999894f...,preprod_int01,tchap.beta.gouv.fr,Chrome Mac OS,Web,1


In [10]:
# Sauvegarder les résultats dans un fichier Parquet
ddf_grouped.to_parquet('user_daily_visits.parquet')

In [12]:


# 1. Vérification et conversion en Pandas DataFrame si nécessaire
if isinstance(ddf_grouped, dd.DataFrame):
    # Si le DataFrame est toujours un Dask DataFrame, on le convertit en Pandas pour l'enregistrer dans PostgreSQL
    df_grouped = ddf_grouped.compute()
else:
    df_grouped = ddf_grouped  # Si c'est déjà un Pandas DataFrame, pas besoin de le reconvertir

# Conversion de 'month' en datetime (par exemple, le premier jour du mois)
df_grouped['month'] = df_grouped['month'].dt.to_timestamp()

# 2. Sauvegarder le DataFrame dans PostgreSQL dans la table 'user_daily_visits_agg_month'
df_grouped.to_sql('user_daily_visits_agg_month', con=engine, if_exists='replace', index=False, chunksize=10000, method='multi')


172